# DataCo deterministic preprocessing

## 1. Objective

Build a reproducible, non-learned cleaning foundation for the DataCo supply-chain data. This notebook performs no EDA, plotting, splitting, feature fitting, or modeling.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_raw_data, raw_file_metadata
from src.leakage_guardrails import (
    EDA_ONLY_COLUMNS, FORBIDDEN_MODEL_COLUMNS, IDENTIFIER_COLUMNS,
    PREDICTION_SAFE_CANDIDATES, REVIEW_REQUIRED_COLUMNS, TARGET_COLUMNS,
)
from src.preprocessing import (
    clean_supply_chain_data, parse_dates, remove_exact_duplicates,
    save_processed_data, standardize_column_names, summarize_missingness,
)

## 2. Dataset source and integrity

The raw file comes from the DataCo Smart Supply Chain for Big Data Analysis dataset. It is read with Latin-1 encoding and treated as immutable. Only non-sensitive file metadata is displayed.

In [ ]:
integrity = raw_file_metadata()
raw = load_raw_data()
{**integrity, 'raw_rows': raw.shape[0], 'raw_columns': raw.shape[1]}

## 3. Unit of analysis

Rows are order items, not independent orders. A repeated Order ID can represent multiple legitimate products in one order and must not be removed merely because it repeats.

In [ ]:
{
    'rows': len(raw),
    'unique_order_ids': raw['Order Id'].nunique(),
    'repeated_order_id_rows': raw['Order Id'].duplicated().sum(),
}

## 4. Column-name normalization and data types

Names are deterministically converted to snake_case. A normalization collision raises an error rather than overwriting a column.

In [ ]:
normalized = standardize_column_names(raw)
normalized.dtypes.rename('dtype').to_frame()

## 5. Missing-value summary

Missing values are reported but not globally dropped or statistically imputed. Learned imputation belongs inside a future training-only pipeline.

In [ ]:
missing_summary = summarize_missingness(normalized)
missing_summary.loc[missing_summary['missing_count'] > 0]

## 6. Exact duplicates versus repeated orders

Only exact full-row duplicates are removed. Repeated `order_id` values remain because the dataset is at order-item grain.

In [ ]:
deduplicated, exact_duplicate_count = remove_exact_duplicates(normalized)
{
    'exact_duplicates_removed': exact_duplicate_count,
    'rows_after_exact_duplicate_removal': len(deduplicated),
    'unique_order_ids': deduplicated['order_id'].nunique(),
}

## 7. Date conversion

Both order and shipping timestamps are converted. Failed conversions become `NaT` and are counted; rows are not silently discarded.

In [ ]:
dated, date_conversion = parse_dates(deduplicated)
date_conversion

## 8. PII and unusable fields

Direct customer names, email, password-like data, and street address are removed without displaying their values. `product_description` is removed only because it is confirmed to be 100% missing. Geographic categories are retained for later policy review.

In [ ]:
cleaned, metadata = clean_supply_chain_data(raw)
{
    'raw_shape': metadata['original_shape'],
    'cleaned_shape': metadata['cleaned_shape'],
    'removed_columns': metadata['removed_columns'],
    'exact_duplicates_removed': metadata['exact_duplicates_removed'],
    'unique_order_ids': metadata['unique_order_ids'],
}

## 9. Leakage-risk classification

Outcome and post-shipment fields are available for data-quality or descriptive work but are forbidden as predictors. Identifiers remain for structure and grouping, not as ordinary model features. Unclear variables require an explicit timing review.

In [ ]:
{
    'target': sorted(TARGET_COLUMNS),
    'eda_only': sorted(EDA_ONLY_COLUMNS),
    'forbidden_model_columns': sorted(FORBIDDEN_MODEL_COLUMNS),
    'identifiers_for_structure_not_modeling': sorted(IDENTIFIER_COLUMNS),
    'prediction_safe_candidates': sorted(PREDICTION_SAFE_CANDIDATES),
    'review_required_count': len(REVIEW_REQUIRED_COLUMNS),
}

## 10. Save the reproducible local output

The cleaned file is generated under `data/processed/` and ignored by Git. It can always be rebuilt from the immutable raw CSV and repository code.

In [ ]:
processed_path = save_processed_data(cleaned)
{
    'processed_path': str(processed_path.relative_to(PROJECT_ROOT)),
    'final_rows': len(cleaned),
    'final_columns': len(cleaned.columns),
    'target_retained': 'late_delivery_risk' in cleaned.columns,
    'order_id_retained': 'order_id' in cleaned.columns,
}

## 11. Decision summary

The workflow normalizes names and whitespace, validates schema and target values, converts dates, removes exact duplicates, removes the all-missing product description and documented direct PII, and preserves legitimate missing values and repeated order IDs. Outliers remain unless proven invalid. No learned preprocessing is fitted globally.